# 2.2 — Perform Data Analysis Given a Use Case

**Exam domain:** Gen AI Functions (Domain 2.0) · **Weight:** 38%

## The problem this solves

Half the answers a business wants are in tables, and half are in PDFs nobody has read since they were
signed. "Which product category has the most negative reviews?" needs the reviews, which are documents,
and the product hierarchy, which is a table. Historically that meant two teams, two systems and a
quarterly reconciliation meeting.

This notebook covers the two Snowflake services that take each half — Cortex Search for the documents,
Cortex Analyst for the tables — and the point where they meet.

## What you will be able to do

- Pull structure out of PDFs with `AI_PARSE_DOCUMENT` and `AI_EXTRACT`, and say which one a task needs
- Chunk and index text into a Cortex Search service, and query it with filters and tuned ranking
- Model a table as a semantic view so Cortex Analyst can answer questions about it in SQL
- Steer Analyst with verified queries and custom instructions
- Read the account-usage views that tell you what all of this is costing

## Before you start

- Run `setup/dataset.sql`. It creates `GENAI_STUDY.PUBLIC.SUPPORT_TICKETS` and `GENAI_STUDY.PUBLIC.PRODUCTS`.
- Stage the files in `sample_docs/` to `@GENAI_STUDY.PUBLIC.DOCS_STAGE` for the document examples.
- Creating a search service needs `CREATE CORTEX SEARCH SERVICE` on the schema, `SELECT` on the base
  tables and `USAGE` on a warehouse; creating a semantic view needs `CREATE SEMANTIC VIEW` on the schema.
- Notebook 2.1 covers the individual AI functions used here.

📖 **Snowflake documentation for this notebook**
- [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)
- [CREATE CORTEX SEARCH SERVICE](https://docs.snowflake.com/en/sql-reference/sql/create-cortex-search)
- [Query a Cortex Search service](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/query-cortex-search-service)
- [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)
- [CREATE SEMANTIC VIEW](https://docs.snowflake.com/en/sql-reference/sql/create-semantic-view)
- [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)


---

## Which half of the problem are you solving?

The tool follows the shape of the data, not the shape of the question.

| Data | Reach for | Typical questions |
|---|---|---|
| **Unstructured** — contracts, invoices, e-mails, PDFs | `AI_PARSE_DOCUMENT`, `AI_EXTRACT`, `AI_EMBED`, Cortex Search | "What does the MSA say about termination?" "Find tickets about compromised credentials" |
| **Structured** — tables, views | Cortex Analyst over a semantic view, plus `AI_AGG` and `AI_SUMMARIZE_AGG` | "Revenue by region last quarter" "Which category grew fastest?" |
| **Both at once** | A Cortex Agent holding a Search tool and an Analyst tool | "Which product category has the most negative reviews?" |

---
## Part A — Unstructured data

### Parse, or extract, or both

`AI_EXTRACT` reads a FILE directly. You do **not** need to parse a document first to pull fields out of
it. Parse first only when you want the document's full text for some other purpose — most often
chunking it for retrieval.

| You want | Use |
|---|---|
| A handful of named fields from a document | `AI_EXTRACT(file => TO_FILE(...), responseFormat => ...)` |
| The whole document as text or markdown | `AI_PARSE_DOCUMENT(file, {'mode': 'LAYOUT'})` |
| Fields *and* the full text | `AI_PARSE_DOCUMENT` once, then `AI_EXTRACT` over the parsed string |

`AI_PARSE_DOCUMENT` options:

| Option | Values | Notes |
|---|---|---|
| `mode` | `'OCR'` (default), `'LAYOUT'` | OCR gives plain text; LAYOUT gives markdown that preserves tables and headings |
| `page_split` | TRUE / FALSE (default FALSE) | TRUE returns `{"pages":[{"content": ..., "index": 0}, ...]}` — indexes start at 0. Supported for PDF, PPTX and DOCX |
| `page_filter` | ARRAY of `{start, end}` objects | Restricts which pages are processed. There is no `page_limit` option |
| `extract_images` | TRUE / FALSE | Adds an `images` array with bounding boxes and base64 data; requires LAYOUT mode |

Without `page_split`, the result is `{"content": "..."}` — read it as `:content::VARCHAR`.

→ [More on AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document) ·
[More on AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)


In [ ]:
%%sql -r doc_parse_extract_1
-- ============================================================
-- Option A, fewest steps: AI_EXTRACT reads the FILE directly.
-- AI_PARSE_DOCUMENT is not a prerequisite for AI_EXTRACT.
-- ============================================================
SELECT
    AI_EXTRACT(
        file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf'),
        responseFormat => {
            'invoice_number': 'What is the invoice or document reference number?',
            'total_amount':   'What is the total amount due?',
            'vendor_name':    'What company or vendor issued this invoice?'
        }
    ):response AS invoice_fields;        -- NOTE the :response wrapper


In [ ]:
%%sql -r doc_parse_extract_2
-- ============================================================
-- Option B: parse first, when you also want the raw markdown or text --
-- for example to chunk it for Cortex Search.
-- ============================================================
SELECT
    AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf'),
        {'mode': 'LAYOUT'}       -- 'OCR' (default) = text only; 'LAYOUT' = markdown with tables/headers
    ) AS parsed_doc;             -- -> {"content": "..."}


In [ ]:
%%sql -r doc_parse_extract_3
WITH doc AS (
    SELECT AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf'),
        {'mode': 'LAYOUT'}
    ):content::VARCHAR AS doc_text
)
SELECT
    AI_EXTRACT(
        doc_text,
        {'invoice_number': 'What is the invoice number?',
         'total_amount':   'What is the total amount due?',
         'vendor_name':    'What company or vendor issued this invoice?'}
    ):response AS invoice_fields
FROM doc;

-- Options: mode | page_split | page_filter | extract_images
--   page_filter is an ARRAY of {start, end} objects; there is no page_limit option
--   page_split TRUE  -> {"pages":[{"content":"...","index":0}, ...]}  (indexes start at 0)
--   extract_images requires LAYOUT mode


> ### ⚠️ Common misconceptions
>
> **"`AI_PARSE_DOCUMENT` is step one of every document pipeline."**
> It is step one of a *retrieval* pipeline, where you need the text to chunk. For field extraction
> `AI_EXTRACT` takes the FILE itself, and parsing first is a second billed pass over every page of
> every document for output you then throw away.
> → [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)
>
> **"I will limit the call to the first three pages with `page_limit`."**
> There is no `page_limit` option. The option is `page_filter`, and it takes an ARRAY of `{start, end}`
> objects. An unrecognised key does not raise an error that names it, so the call runs over the whole
> document and bills accordingly.
> → [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
>
> **"OCR mode and LAYOUT mode differ only in quality."**
> They differ in *structure*. OCR returns a flat string, so a financial table arrives as a run of
> numbers with no column alignment. LAYOUT returns markdown, so the table is still a table when the
> next model reads it. LAYOUT is also what `extract_images` requires.
> → [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)


---

### Chunking: the step that decides how good your retrieval is

An embedding model has a context window — 512 tokens for the Arctic-embed family. Text longer than that
cannot be embedded in one piece, and even when a model *can* take a long document, embedding the whole
thing averages every idea in it into one vector. Retrieval then returns the document, not the passage,
and the answering model has to find the relevant sentence itself.

So you split first. The two dials are chunk size and overlap, and they trade against each other:

- **Smaller chunks** retrieve more precisely and ground the answer more tightly. They also lose the
  context around a sentence, and produce more rows to embed, store and search.
- **Overlap** stops an idea being cut in half at a boundary, at the cost of storing the overlapping
  text more than once.

Measure before you split: `AI_COUNT_TOKENS('AI_EMBED', '<model>', text)` tells you which rows are over
the limit, so you only pay to chunk the ones that need it.

→ [More on chunking](https://docs.snowflake.com/en/sql-reference/functions/split_text_recursive_character-snowflake-cortex)


In [ ]:
%%sql -r chunking_embed_1
-- Chunking strategy before RAG embedding
-- AI_COUNT_TOKENS takes the function name first; SPLIT_TEXT_RECURSIVE_CHARACTER is
-- namespaced under SNOWFLAKE.CORTEX and needs a `format` argument.

-- Step 1: which tickets exceed the embedding model's 512-token context?
SELECT
    ticket_id,
    AI_COUNT_TOKENS('AI_EMBED', 'snowflake-arctic-embed-m-v1.5', ticket_text) AS tokens,
    IFF(AI_COUNT_TOKENS('AI_EMBED', 'snowflake-arctic-embed-m-v1.5', ticket_text) > 512,
        'CHUNK', 'EMBED_DIRECT') AS strategy
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;


In [ ]:
%%sql -r chunking_embed_2
-- Step 2: chunk and embed
SELECT
    t.ticket_id,
    f.index          AS chunk_index,
    f.value::VARCHAR AS chunk_text,
    AI_EMBED('snowflake-arctic-embed-m-v1.5', f.value::VARCHAR) AS embedding
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS t,
     LATERAL FLATTEN(
         SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
             t.ticket_text, 'none', 400, 50, ['\n\n', '. ', ' ']
         )
     ) f;

-- Snowflake's guidance: keep chunks <= 512 tokens (~385 English words) even when the model
-- supports a longer context — smaller chunks retrieve more precisely and ground the LLM better.


---

### Cortex Search — retrieval you do not have to build

You could build search from parts you already know: `AI_EMBED` into a `VECTOR` column, then
`VECTOR_COSINE_SIMILARITY` at query time. Cortex Search is the managed version of that, and it does
three things in one service:

- **Vector search** for semantically similar text
- **Keyword search** for lexically similar text — which is what catches exact error codes and product
  SKUs that embeddings blur
- **Semantic reranking** over the combined result set

It also keeps itself up to date. `TARGET_LAG` says how far behind the base data the index may fall, and
the service refreshes on that schedule — which is why **change tracking must be enabled, with non-zero
time-travel retention, on every underlying object**.

```sql
CREATE [ OR REPLACE ] CORTEX SEARCH SERVICE <name>
  ON <search_column>
  [ PRIMARY KEY ( <col> [, ...] ) ]
  ATTRIBUTES <col> [, ...]
  WAREHOUSE = <warehouse>
  TARGET_LAG = '<num> { seconds | minutes | hours | days }'
  [ EMBEDDING_MODEL = <model> ]
  [ REFRESH_MODE = { FULL | INCREMENTAL } ]
  [ INITIALIZE = { ON_CREATE | ON_SCHEDULE } ]
  [ AUTO_SUSPEND = <seconds> ]
  [ COMMENT = '<comment>' ]
AS <query>;
```

- `ON` names the one column that is indexed and searched.
- `ATTRIBUTES` names the columns you may **filter** on later. A column you forget here cannot be
  filtered without recreating the service.
- `EMBEDDING_MODEL` defaults to `snowflake-arctic-embed-m-v1.5`.
- `AUTO_SUSPEND` is in seconds, minimum `1800`.

A second form indexes several columns at once with `TEXT INDEXES` and `VECTOR INDEXES`, queried through
a `multi_index_query` block.

**What it costs:** warehouse compute to refresh the service and run the source query, embedding tokens
for every row that changes, serving compute charged **per GB per month of indexed data whether or not
anyone queries it**, and storage. The idle cost is the one that surprises people.

→ [More on Cortex Search](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview) ·
[More on the DDL](https://docs.snowflake.com/en/sql-reference/sql/create-cortex-search)


In [ ]:
%%sql
-- ============================================================
-- Cortex Search service over support tickets
-- Prerequisites for the CREATING role:
--   * SNOWFLAKE.CORTEX_USER or SNOWFLAKE.CORTEX_EMBED_USER
--   * CREATE CORTEX SEARCH SERVICE on the schema
--   * SELECT on the base tables, USAGE on the warehouse
--   * CHANGE TRACKING enabled on the base objects
-- ============================================================
ALTER TABLE GENAI_STUDY.PUBLIC.SUPPORT_TICKETS SET CHANGE_TRACKING = TRUE;

CREATE OR REPLACE CORTEX SEARCH SERVICE GENAI_STUDY.PUBLIC.TICKET_SEARCH
    ON safe_text                                  -- the single column that is indexed and searched
    PRIMARY KEY (ticket_id)                       -- optional; enables optimized incremental refresh
    ATTRIBUTES ticket_id, category, status, language, priority   -- the only filterable columns
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-m-v1.5'   -- default if omitted
    AUTO_SUSPEND = 1800                           -- seconds; minimum 1800
    COMMENT = 'Hybrid search over redacted support tickets'
AS
    SELECT
        ticket_id,
        AI_REDACT(ticket_text) AS safe_text,
        category, status, language, priority
    FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;


In [ ]:
%%sql -r cortex_search_setup_2
-- ============================================================
-- Query it (hybrid: text/BM25 + vector, then semantic reranking)
-- ============================================================
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'GENAI_STUDY.PUBLIC.TICKET_SEARCH',
        '{
            "query": "unauthorized account access security breach",
            "columns": ["ticket_id", "category", "safe_text"],
            "filter": {"@eq": {"status": "open"}},
            "limit": 5
        }'
    )
)['results'] AS results;

-- Tune ranking with scoring_config (weights + boost functions), or turn reranking off for latency:
--   "scoring_config": {
--       "weights":   {"texts": 1, "vectors": 4, "reranker": 1},
--       "functions": {"time_decays":   [{"column": "created_at", "weight": 2, "limit_hours": 720}],
--                     "numeric_boosts":[{"column": "priority_rank", "weight": 1}]}
--   }
--   "scoring_config": {"reranker": "none"}                -- disable reranking for latency

-- Multi-index services index several columns with different index types:
--   CREATE CORTEX SEARCH SERVICE ...
--     TEXT INDEXES (product_name, sku)
--     VECTOR INDEXES (description MODEL = 'snowflake-arctic-embed-l-v2.0', user_supplied_vec)
--   ... then query with "multi_index_query": {"product_name": [{"text": "sparkle"}], ...}

-- Governance: the service searches with owner's rights. A querying role needs USAGE on the
-- service plus USAGE on its database and schema, and then sees whatever the owner can see.


> ### ⚠️ Common misconceptions
>
> **"A user who cannot read the base table cannot see its rows through the search service."**
> They can. Cortex Search services **search with owner's rights**: a querying role needs `USAGE` on the
> service, its database and its schema, and then sees whatever the *owner* can see. This is why the
> service in this notebook is built over `AI_REDACT(ticket_text)` rather than the raw column — the
> redaction, not the grant, is what keeps personal data out of the results.
> → [Query a Cortex Search service](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/query-cortex-search-service)
>
> **"Adding a filter on a column I selected in the service query will just work."**
> Only columns listed in `ATTRIBUTES` are filterable. Filtering on anything else fails, and the fix is
> to recreate the service — which means re-embedding every row. Decide your filter columns before you
> build.
> → [CREATE CORTEX SEARCH SERVICE](https://docs.snowflake.com/en/sql-reference/sql/create-cortex-search)
>
> **"Change tracking is an optimisation I can turn on later."**
> It is a prerequisite. Change tracking with non-zero time-travel retention must be enabled on all
> underlying objects for the service to refresh incrementally.
> → [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)


> ### 🤔 Stop and think
>
> - A Cortex Search service bills serving compute per GB of indexed data every month, whether or not
>   anyone searches. For a corpus that gets queried twice a week, is that better or worse than embedding
>   into a table yourself and running `VECTOR_COSINE_SIMILARITY` on demand — and what do you give up?
> - Owner's rights means the service can return text the querying role could never `SELECT`. Who in your
>   organisation would have to approve a service built over a table with customer names in it, and what
>   would you show them to make the risk concrete?
> - A verified query steers Analyst towards SQL a human has already checked. It also freezes a business
>   definition in place. Who notices when "active customer" changes meaning, and how does the semantic
>   view find out?


---
## Part B — Structured data with Cortex Analyst

Cortex Analyst turns a natural-language question into SQL. The thing that makes it reliable is not the
model but the **semantic model** you give it: a description of what your tables *mean*, in business
terms, so that "revenue" maps to a specific expression rather than to whichever column has REV in the
name.

```
User question -> Cortex Analyst -> generated SQL -> Snowflake -> answer
```

### The pieces

| Component | What it is |
|---|---|
| **Semantic view** *(recommended)* | A schema-level Snowflake object holding logical tables, relationships, facts, dimensions, metrics and synonyms. Governed by ordinary RBAC, and shareable |
| **YAML semantic model on a stage** *(legacy)* | The same idea as a file. Still supported for backward compatibility; access is governed by stage permissions instead of object grants |
| **Verified queries** | Question-and-SQL pairs a human has checked, which steer generation |
| **Custom instructions** | Free-text rules, split into SQL generation and question categorization |
| **Cortex Search integration** | Attach a search service to a dimension so Analyst can resolve literal values — matching "Acme Corp." to the stored "ACME" |

A *fact* is a row-level numeric column, a *dimension* is something you group or filter by, and a
*metric* is an aggregation over a fact. Declaring `SUM(monthly_revenue)` once as a metric is what stops
two users getting two different revenue numbers.

### Required privileges

```sql
-- one of these two database roles
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER         TO ROLE <role>;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE <role>;
-- plus SELECT on the tables behind the model, USAGE on any referenced Cortex Search service,
-- and READ/WRITE on the stage if the model is a YAML file
```

`CORTEX_ANALYST_USER` is the narrower of the two: it grants Analyst and nothing else.

### What it costs

Cortex Analyst is billed **per message processed** — successful responses — **not per token**. Token
counts only enter the picture when Analyst is invoked through a Cortex Agent. Executing the generated
SQL costs warehouse credits separately. Track messages in
`SNOWFLAKE.ACCOUNT_USAGE.CORTEX_ANALYST_USAGE_HISTORY`, whose columns are `START_TIME`, `END_TIME`,
`REQUEST_COUNT`, `CREDITS` and `USERNAME`.

→ [More on Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)

### Writing the semantic view

Clause order is fixed: `TABLES` → `RELATIONSHIPS` → `FACTS` → `DIMENSIONS` → `METRICS`. Every expression
is written **alias first**: `products.total_revenue AS SUM(products.monthly_revenue)`. You must define
at least one dimension or metric.

`CREATE OR ALTER SEMANTIC VIEW` lets you add, remove and modify parts of an existing view without
dropping it — which matters, because a semantic view accumulates verified queries you do not want to
lose.

→ [More on CREATE SEMANTIC VIEW](https://docs.snowflake.com/en/sql-reference/sql/create-semantic-view)


In [ ]:
%%sql
-- ============================================================
-- CREATE SEMANTIC VIEW
-- Clause order is fixed: TABLES -> RELATIONSHIPS -> FACTS -> DIMENSIONS -> METRICS
-- Each expression is written  <alias> AS <expression>  (alias first).
-- ============================================================
CREATE OR REPLACE SEMANTIC VIEW GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC
    TABLES (
        products AS GENAI_STUDY.PUBLIC.PRODUCTS
            PRIMARY KEY (product_id)
            WITH SYNONYMS ('catalog', 'offerings')
            COMMENT = 'Product catalog with revenue and ratings'
    )
    FACTS (
        products.monthly_revenue AS monthly_revenue COMMENT 'Revenue for one product in one month',
        products.units_sold      AS units_sold      COMMENT 'Units sold for one product',
        products.rating          AS rating          COMMENT 'Customer rating out of 5'
    )
    DIMENSIONS (
        products.product_name AS product_name COMMENT 'Name of the product',
        products.category     AS category     WITH SYNONYMS ('product type') COMMENT 'Product category',
        products.region       AS region       COMMENT 'Sales region',
        products.launch_date  AS launch_date  COMMENT 'Date the product launched',
        products.is_active    AS is_active    COMMENT 'Whether the product is currently sold'
    )
    METRICS (
        products.total_monthly_revenue AS SUM(products.monthly_revenue) COMMENT 'Total monthly revenue',
        products.avg_rating            AS AVG(products.rating)          COMMENT 'Average customer rating',
        products.total_units_sold      AS SUM(products.units_sold)      COMMENT 'Total units sold'
    )
    COMMENT = 'Semantic model powering Cortex Analyst for the product catalog';


In [ ]:
%%sql -r semantic_view_create_2
SHOW SEMANTIC VIEWS IN SCHEMA GENAI_STUDY.PUBLIC;


In [ ]:
%%sql -r semantic_view_create_3
DESCRIBE SEMANTIC VIEW GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC;


In [ ]:
# Cortex Analyst REST API call — ask a natural language question
import json, requests

# In Snowflake Notebooks, use the session token directly
from snowflake.snowpark.context import get_active_session
session = get_active_session()

payload = {
    "messages": [
        {"role": "user", "content": [{"type": "text", "text": "Which product has the highest monthly revenue?"}]}
    ],
    "semantic_model_file": "@GENAI_STUDY.PUBLIC.DOCS_STAGE/products_semantic.yaml"
    # OR: "semantic_view": "GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC"
}

# The REST endpoint — replace <account> with your Snowflake account identifier
ANALYST_URL = "https://<account>.snowflakecomputing.com/api/v2/cortex/analyst/message"

print("Payload structure for Cortex Analyst:")
print(json.dumps(payload, indent=2))
print("\nIn production: POST this to the Analyst REST endpoint with a valid JWT or OAuth token.")
print("In Snowflake Notebooks: use SNOWFLAKE.CORTEX.ANALYST directly if available,")
print("or invoke via Streamlit app (see 2.3.ipynb).")

In [ ]:
%%sql
-- ============================================================
-- VERIFIED QUERIES & CUSTOM INSTRUCTIONS
-- There is no ALTER SEMANTIC VIEW ... ADD VERIFIED QUERY statement. They are declared
-- inside CREATE (or CREATE OR ALTER) SEMANTIC VIEW:
--   AI_VERIFIED_QUERIES ( <name> AS ( QUESTION '...' SQL '...' ) )
--   AI_SQL_GENERATION '<instructions>'            -- note: no '=' before the string
--   AI_QUESTION_CATEGORIZATION '<instructions>'
-- CREATE OR ALTER adds, removes or modifies them without dropping the view.
-- ============================================================
CREATE OR ALTER SEMANTIC VIEW GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC
    TABLES (
        products AS GENAI_STUDY.PUBLIC.PRODUCTS PRIMARY KEY (product_id)
    )
    FACTS (
        products.monthly_revenue AS monthly_revenue
    )
    DIMENSIONS (
        products.region    AS region,
        products.is_active AS is_active
    )
    METRICS (
        products.total_monthly_revenue AS SUM(products.monthly_revenue)
    )
    AI_SQL_GENERATION 'Filter is_active = TRUE unless the user explicitly asks about inactive products. Round currency to 2 decimals.'
    AI_QUESTION_CATEGORIZATION 'Reject questions about individual customers; direct the user to the CRM team.'
    AI_VERIFIED_QUERIES (
        revenue_by_region AS (
            QUESTION 'What is the total monthly revenue by region?'
            SQL 'SELECT region, SUM(monthly_revenue) AS total_revenue FROM GENAI_STUDY.PUBLIC.PRODUCTS GROUP BY region ORDER BY total_revenue DESC'
        )
    );


In [ ]:
%%sql -r analyst_vqr_2
-- Equivalent keys in the legacy YAML semantic model:
--   verified_queries:
--   module_custom_instructions:
--       sql_generation: |
--         "..."
--       question_categorization: |
--         "..."
--   (the older flat `custom_instructions:` key still works)

DESCRIBE SEMANTIC VIEW GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC;


> ### ⚠️ Common misconceptions
>
> **"Passing a `semantic_models` array lets Analyst join across all of them in one query."**
> It does not. For each query, Cortex Analyst chooses the most appropriate model or view from the list
> and generates SQL against **that one**. A question that genuinely spans two models is not answered by
> listing both — it needs a single model that contains both sets of tables, or an agent that calls
> Analyst more than once.
> → [Cortex Analyst REST API](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/rest-api)
>
> **"Each physical table can appear only once in a semantic model, so I cannot model both a ship-to and
> a bill-to region."**
> Logical table **names** must be unique, but the same physical table may appear several times under
> different logical names pointing at the same `base_table`. That is exactly how you model role-playing
> dimensions — `customer_region` and `supplier_region` both over `region`.
> → [Semantic view YAML specification](https://docs.snowflake.com/en/user-guide/views-semantic/semantic-view-yaml-spec)
>
> **"I will add a verified query later with `ALTER SEMANTIC VIEW ... ADD VERIFIED QUERY`."**
> There is no such statement. Verified queries and custom instructions are declared inside
> `CREATE SEMANTIC VIEW` or `CREATE OR ALTER SEMANTIC VIEW`, as `AI_VERIFIED_QUERIES`,
> `AI_SQL_GENERATION` and `AI_QUESTION_CATEGORIZATION`. Note that the two instruction clauses take their
> string directly, with no `=` sign.
> → [CREATE SEMANTIC VIEW](https://docs.snowflake.com/en/sql-reference/sql/create-semantic-view)
>
> **"Analyst is billed per token like the AI functions, so short questions are cheap."**
> It is billed per message. A one-word follow-up costs the same as a paragraph. The lever on Analyst
> cost is the number of round trips, not their length — which is a reason to give users suggested
> questions rather than letting them iterate blindly.
> → [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)


---
## Part C — Making it fast, accurate and affordable

| Factor | Guidance |
|---|---|
| **Latency** | Smaller models answer faster. Start with the smallest model that clears your accuracy bar and move up only when a measurement says to |
| **Accuracy** | Larger models generalise better on open-ended work. On a narrow, well-labelled task, fine-tuning a small model can close the gap for less money at inference time |
| **Grounding** | `temperature` is already 0 by default. Beyond that, constrain the output with a `response_format` schema and give the model retrieved text to work from rather than asking it to recall. `guardrails => TRUE` filters unsafe output |
| **Provisioned Throughput** | Reserves inference capacity in provisioned throughput units (PTUs) for a **one-month term**, billed in credits per PTU per hour **whether or not you use them**, and it does not renew automatically. Minimum PTUs are per model: 64 for Llama 3.1-8B, 128 for Llama 3.1-70B and Snowflake-Llama3.3-70B, 256 for Mistral Large 2, 512 for Llama 3.1-405B and Snowflake-Llama3.1-405B |
| **Model availability** | Model names are not universal. `SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS` reports `lifecycle_status` (GA, PUPR, PRPR, LEGACY, EOL), `in_region_availability`, `cross_region_availability`, `legacy_date` and `eol_date`, filtered to what your role may use |
| **Batch shape** | Filter and truncate before the AI call, not after. Store results instead of recomputing them. Prefer a task-specific function over free-form `AI_COMPLETE` when the output is structured |

The escalation ladder that keeps cost down: **task-specific function → small general model → large
general model → fine-tuned small model → provisioned throughput**. Most workloads stop at step two, and
every step up should be justified by a measurement rather than a hunch.

→ [More on Provisioned Throughput](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput) ·
[More on SHOW CORTEX BASE MODELS](https://docs.snowflake.com/en/sql-reference/sql/show-cortex-base-models)

### Where the bill actually appears

| View | What it holds |
|---|---|
| `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY` | All Cortex AI functions **including** `AI_PARSE_DOCUMENT`, with credits. Columns: `START_TIME`, `END_TIME`, `FUNCTION_NAME`, `MODEL_NAME`, `QUERY_ID`, `WAREHOUSE_ID`, `ROLE_NAMES`, `QUERY_TAG`, `USER_ID`, `METRICS`, `CREDITS`, `IS_COMPLETED`. Updated every ~2 minutes, 5-minute SLA |
| `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY` | Token-level detail in one-hour buckets. **Excludes** `AI_PARSE_DOCUMENT`, which is metered per page rather than per token |
| `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_ANALYST_USAGE_HISTORY` | Analyst messages and credits |

`CORTEX_FUNCTIONS_USAGE_HISTORY` still exists but is no longer updated; its replacement is
`CORTEX_AISQL_USAGE_HISTORY`.

→ [More on Cortex AI function cost management](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management)


In [ ]:
%%sql -r model_comparison_1
-- Which base models can this role use, and in which regions?
SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS;


In [ ]:
%%sql -r model_comparison_2
-- Compare a small vs. a large model on the same prompt
SELECT
    'llama3.1-8b' AS model,
    AI_COMPLETE('llama3.1-8b', 'Classify as billing/technical/shipping: ' || ticket_text) AS result
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS WHERE ticket_id = 1001
UNION ALL
SELECT
    'llama3.3-70b',
    AI_COMPLETE('llama3.3-70b', 'Classify as billing/technical/shipping: ' || ticket_text)
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS WHERE ticket_id = 1001;


In [ ]:
%%sql -r perf_monitoring_1
-- ============================================================
-- USAGE MONITORING
-- SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FUNCTIONS_USAGE_HISTORY is no longer updated. Use:
--   CORTEX_AI_FUNCTIONS_USAGE_HISTORY  -> all AI functions INCLUDING AI_PARSE_DOCUMENT; credits
--   CORTEX_AISQL_USAGE_HISTORY         -> token-level detail; EXCLUDES AI_PARSE_DOCUMENT
-- ============================================================

-- Credits by function and model, last 7 days
SELECT
    DATE_TRUNC('day', START_TIME) AS day,
    FUNCTION_NAME,
    MODEL_NAME,
    COUNT(DISTINCT QUERY_ID)      AS call_count,
    SUM(CREDITS)                  AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3
ORDER BY total_credits DESC;


In [ ]:
%%sql -r perf_monitoring_2
-- Columns: START_TIME, END_TIME, FUNCTION_NAME, MODEL_NAME, QUERY_ID, WAREHOUSE_ID,
--          ROLE_NAMES, QUERY_TAG, USER_ID, METRICS, CREDITS, IS_COMPLETED
-- Refreshed every ~2 minutes (5-minute SLA).

-- Token-level detail (1-hour buckets, keyed on query completion time)
SELECT
    DATE_TRUNC('day', USAGE_TIME) AS day,
    FUNCTION_NAME,
    MODEL_NAME,
    SUM(TOKENS)                   AS total_tokens,
    SUM(TOKEN_CREDITS)            AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AISQL_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3
ORDER BY total_credits DESC;


In [ ]:
%%sql -r perf_monitoring_3
-- Columns: USAGE_TIME, MODEL_NAME, FUNCTION_NAME, TOKEN_CREDITS, TOKENS,
--          TOKEN_CREDITS_GRANULAR, TOKENS_GRANULAR, QUERY_ID, QUERY_TAG, USER_ID, WAREHOUSE_ID

-- Cortex Analyst is billed per message processed, not per token:
SELECT START_TIME, REQUEST_COUNT, CREDITS, USERNAME
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_ANALYST_USAGE_HISTORY
ORDER BY START_TIME DESC LIMIT 20;


---
## Putting it together

**Scenario.** A retailer has 50,000 product-review PDFs on a stage, a structured `PRODUCTS` table, and
users who want to ask *"Which product category has the most negative reviews?"*

**Design the end-to-end architecture. Which components, in what order?**

### Worked solution

1. **Extract** — `AI_EXTRACT(file => TO_FILE(...), responseFormat => ...)` straight from the stage, or
   `AI_PARSE_DOCUMENT(..., {'mode':'LAYOUT'})` when you also need the full markdown for chunking.
2. **Land** the review text in a `REVIEWS` table and enable `CHANGE_TRACKING`.
3. **Chunk** with `SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(text, 'none', 512, 50)`, sized to the
   embedding model's context window.
4. **Index** — `CREATE CORTEX SEARCH SERVICE` over the chunks, with a `PRIMARY KEY`, a `TARGET_LAG` you
   can live with, and the filter columns declared as `ATTRIBUTES`.
5. **Model** the structured side — `CREATE SEMANTIC VIEW` over `PRODUCTS`, with `AI_VERIFIED_QUERIES`
   for the questions you already know people ask. Attaching a Cortex Search service to the `category`
   dimension lets Analyst resolve messy literal values.
6. **Orchestrate** — a **Cortex Agent** with two tools, Search for the unstructured half and Analyst for
   the structured half. The agent plans, calls tools, reflects on the results, and answers.
7. **Surface** — Snowflake CoWork, a Streamlit app (2.3), the Agent REST API, or an MCP client.

This is the canonical retrieval-plus-text-to-SQL hybrid agent on Snowflake.

**The cost shape to carry away.** Search costs warehouse refresh + embedding tokens per changed row +
serving per GB per month even at zero queries + storage. Analyst costs per message. AI functions cost
per token, except `AI_PARSE_DOCUMENT`, which is metered per page.


---
## Part D — Audio, images and video

Everything so far assumed text. Three functions get the other modalities into a form the text toolkit
can already handle.

| Modality | Function | Key limits |
|---|---|---|
| **Audio or video to text** | `AI_TRANSCRIBE(audio_file [, options] [, err])` | 700 MB, 120 minutes — 60 with timestamps. Billed at 50 tokens per second of audio with a 10-second minimum. Language is auto-detected across roughly thirty languages |
| **Image or document understanding** | `AI_COMPLETE(model, PROMPT('...{0}...', TO_FILE(...)))`, `AI_CLASSIFY`, `AI_FILTER` on FILE inputs | The model must be multimodal; image size 10 MB for most models and 3.75 MB for Claude models |
| **Image, audio or video to vectors** | `AI_EMBED('voyage-multimodal-3', file)`, `AI_MULTI_EMBED('twelvelabs-marengo-embed-3-0', file)` | `AI_EMBED` images up to 10 MB; `AI_MULTI_EMBED` images up to 5 MB, media up to 4 hours and 6 GB |

**`AI_TRANSCRIBE` options.** `timestamp_granularity` takes `"word"` or `"speaker"`. The result is JSON
with `audio_duration` and `text`, plus a `segments` array carrying `start`, `end`, `text` and, for
speaker granularity, `speaker_label`. Audio formats: AAC, FLAC, M4A, MP3, MP4, OGG, WAV, WEBM. Video:
MKV, MOV, MP4, OGV, WEBM, and the file must carry an audio track. Role: `SNOWFLAKE.CORTEX_USER`.
Availability at the time of writing is AWS US West 2, US East 1 and EU Central 1, plus Azure East US 2 —
check the reference page for your region before you design around it.

**`AI_MULTI_EMBED`** differs from `AI_EMBED` in what it returns: an array of per-segment embeddings,
each with an `embedding_option` (visual, audio, transcription or fused), an `embedding_scope`, and
`start_sec` / `end_sec` boundaries. `AI_EMBED` returns one vector for the whole input. That is the
difference between "find the moment in this video" and "find this video".

> **The pattern to remember:** transcribe, then treat the transcript as text. Classification, sentiment,
> extraction, embedding and Cortex Search all work unchanged from there. Multimodal analytics on
> Snowflake is mostly "get it into text or vectors, then do what you already know."

→ [More on AI_TRANSCRIBE](https://docs.snowflake.com/en/sql-reference/functions/ai_transcribe) ·
[More on AI_MULTI_EMBED](https://docs.snowflake.com/en/sql-reference/functions/ai_multi_embed)


In [ ]:
%%sql -r transcribe_basic
-- Transcribe a call recording. Returns JSON: audio_duration, text (+ segments with granularity).
SELECT AI_TRANSCRIBE(
    TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'support_call.mp3')
) AS transcript;

In [ ]:
%%sql -r transcribe_enriched
-- Speaker-turn timestamps, then feed the transcript straight into the text functions
WITH t AS (
    SELECT AI_TRANSCRIBE(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'support_call.mp3'),
        {'timestamp_granularity': 'speaker'}
    ) AS r
)
SELECT
    r:audio_duration::FLOAT                                   AS seconds,
    r:text::VARCHAR                                           AS full_text,
    AI_CLASSIFY(r:text::VARCHAR,
        ['billing','technical','shipping']):labels[0]::VARCHAR AS topic,
    r:segments                                                AS speaker_segments
FROM t;

In [ ]:
%%sql -r multimodal_complete
-- Image understanding: AI_COMPLETE with a PROMPT() object over one or more FILEs
SELECT AI_COMPLETE(
    'claude-sonnet-5',
    PROMPT('Does the product in image {0} match the description: {1}?',
           TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'product_photo.jpg'),
           'A rack-mounted network appliance with two ethernet ports')
) AS image_verdict;

In [ ]:
%%sql -r multi_embed_video
-- Video and audio search vectors: one embedding per segment and modality
SELECT AI_MULTI_EMBED(
    'twelvelabs-marengo-embed-3-0',
    TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'product_demo.mp4')
) AS multimodal_embeddings;

### `AI_SIMILARITY` — "are these two things about the same subject?"

```sql
AI_SIMILARITY( <input1>, <input2> [, <config_object> ] )   -- {'model': '...'}
```

Returns a float from −1 to 1. Defaults are `snowflake-arctic-embed-l-v2.0` for text and
`voyage-multimodal-3` for images, and it cannot compare text against an image.

**When to use which:**

| Situation | Reach for |
|---|---|
| A one-off pairwise comparison, with nowhere to keep vectors | `AI_SIMILARITY` |
| Repeated search over the same corpus | `AI_EMBED` once into a stored `VECTOR` column, then `VECTOR_COSINE_SIMILARITY` |
| Production retrieval with filters, ranking and reranking | a Cortex Search service |

`AI_SIMILARITY` re-embeds both inputs on every call. That is cheap for a handful of rows and wasteful
across a self-join, where the same document gets embedded once per pair it appears in.

→ [More on AI_SIMILARITY](https://docs.snowflake.com/en/sql-reference/functions/ai_similarity)


In [ ]:
%%sql -r similarity_dedupe
-- Deduplicate: which parsed documents are near-identical to each other?
SELECT
    a.doc_id AS doc_a,
    b.doc_id AS doc_b,
    AI_SIMILARITY(a.doc_text, b.doc_text,
                  {'model': 'snowflake-arctic-embed-l-v2.0'}) AS similarity
FROM GENAI_STUDY.PUBLIC.PARSED_DOCUMENTS a
JOIN GENAI_STUDY.PUBLIC.PARSED_DOCUMENTS b ON a.doc_id < b.doc_id
QUALIFY similarity > 0.95
ORDER BY similarity DESC;

In [ ]:
%%sql -r similarity_images
-- Image similarity: does the delivered photo match the catalogue photo?
SELECT AI_SIMILARITY(
    TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'catalogue_photo.jpg'),
    TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'delivered_photo.jpg')
) AS image_similarity;

### One image, one question

There is a three-argument form of `AI_COMPLETE` for visual question answering:

```sql
AI_COMPLETE( <model>, <predicate>, <file> [, <model_parameters> ] )
```

| | Detail |
|---|---|
| Image formats | `.jpg` `.jpeg` `.png` `.gif` `.webp`; `.bmp` additionally on `pixtral` and `llama4` models |
| Maximum size | 10 MB for most models, 3.75 MB for Claude models, which also cap resolution at 8000 x 8000 |
| Stage requirement | The stage must have server-side encryption; client-side encrypted stages are not supported |

Three ways to put an image in front of a model, and they are not interchangeable:

| Form | Use when |
|---|---|
| `AI_COMPLETE(model, predicate, file)` | **one** image, one question |
| `AI_COMPLETE(model, PROMPT('... {0} ... {1} ...', file_a, file_b))` | **several** files, or files mixed with text columns |
| `AI_CLASSIFY(file, categories)` or `AI_FILTER(predicate, file)` | you want a label or a boolean rather than prose — cheaper, and parseable without a second model |

→ [More on the single-file form](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-file)

## Choosing a model

Four inputs to the decision:

| Input | Question to ask |
|---|---|
| **Capability** | Does the task need multimodal input, a long context, or strong reasoning? |
| **Latency** | Smaller is faster. Start small |
| **Accuracy** | Measure on a labelled sample before assuming you need the largest model |
| **Availability** | `SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS` — check `in_region_availability`, `cross_region_availability`, `lifecycle_status`, `legacy_date` and `eol_date` |

The availability column is the one people skip. A model that works in your development region and not in
production is a deployment failure discovered late, and `lifecycle_status` plus `eol_date` are how you
find out that the model you standardised on is on its way out.


In [ ]:
%%sql -r complete_single_image
-- Visual question answering over one image
SELECT AI_COMPLETE(
    'claude-sonnet-5',
    'Which line item on this invoice has the largest amount, and what is that amount?',
    TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_scan.png')
) AS visual_answer;

In [ ]:
%%sql -r model_shortlist
-- Model shortlist for a task: what is available, current, and reachable from this region?
SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS;

---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** What is the default `EMBEDDING_MODEL` for a Cortex Search service, and what is the minimum
`AUTO_SUSPEND`?

<details><summary>Show answer</summary>

The default embedding model is `snowflake-arctic-embed-m-v1.5`, and `AUTO_SUSPEND` is specified in
seconds with a minimum of `1800` — thirty minutes. Writing `AUTO_SUSPEND = 300` expecting five minutes
fails on both counts: it is below the minimum, and the unit is not what a warehouse's `AUTO_SUSPEND`
habit would suggest.

→ [CREATE CORTEX SEARCH SERVICE](https://docs.snowflake.com/en/sql-reference/sql/create-cortex-search)

</details>

**2.** Name the three retrieval mechanisms Cortex Search combines.

<details><summary>Show answer</summary>

Vector search for semantic similarity, keyword search for lexical similarity, and semantic reranking
over the merged result set. The keyword half is the part people forget, and it is what makes exact
strings — an error code, a SKU, a contract reference — findable when an embedding would blur them into
their neighbours.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>

**3.** Which account-usage view covers `AI_PARSE_DOCUMENT`, and why is it not in the other one?

<details><summary>Show answer</summary>

`CORTEX_AI_FUNCTIONS_USAGE_HISTORY` includes it; `CORTEX_AISQL_USAGE_HISTORY` explicitly excludes it.
`AI_PARSE_DOCUMENT` is metered per **page**, not per token, so it has no row to contribute to a
token-level view. A cost report built only on the AISQL view will understate a document pipeline badly.

→ [Cortex AI function cost management](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management)

</details>

**4.** A search service was created with `ATTRIBUTES category, status`. A query filters on `priority`,
which is in the service's `AS` query but not in `ATTRIBUTES`. What happens, and what is the fix?

<details><summary>Show answer</summary>

The filter is rejected — only `ATTRIBUTES` columns are filterable. The fix is to recreate the service
with `priority` in the attribute list, which means re-embedding every row. The tempting workaround,
filtering in the outer SQL after the search returns, changes the meaning of `limit`: you would be
filtering the top N results rather than searching within the filtered set, so a common value can crowd
out every row you wanted.

→ [CREATE CORTEX SEARCH SERVICE](https://docs.snowflake.com/en/sql-reference/sql/create-cortex-search)

</details>

**5.** This is meant to add a verified query to an existing semantic view. What is wrong?

```sql
ALTER SEMANTIC VIEW PRODUCTS_SEMANTIC
  ADD VERIFIED QUERY 'revenue by region' AS SELECT region, SUM(monthly_revenue) ...;
```

<details><summary>Show answer</summary>

No such statement exists. Verified queries live inside the view definition, in an `AI_VERIFIED_QUERIES`
clause, where each entry is `<name> AS ( QUESTION '...' [ VERIFIED_AT ... ] [ VERIFIED_BY ... ] SQL '...' )`.
Use `CREATE OR ALTER SEMANTIC VIEW` to add one without dropping the view and losing everything else in
it.

→ [CREATE SEMANTIC VIEW](https://docs.snowflake.com/en/sql-reference/sql/create-semantic-view)

</details>

**6.** An Analyst request passes `semantic_models` with a sales view and a support view, and a user asks
a question spanning both. What does Analyst do?

<details><summary>Show answer</summary>

It picks one. For each query, Cortex Analyst chooses the most appropriate model or view from the list
and generates SQL against that single model — it does not join across views. The answer will be about
whichever half it chose, and it will look confident. If questions genuinely span both domains, either
model both sets of tables in one semantic view, or put Analyst behind an agent that can call it more
than once and combine the results.

→ [Cortex Analyst REST API](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/rest-api)

</details>

**7.** You need only the invoice number and total from 10,000 staged PDFs. Is
`AI_PARSE_DOCUMENT` then `AI_EXTRACT` a reasonable pipeline?

<details><summary>Show answer</summary>

No — it is double work. `AI_EXTRACT` accepts a FILE directly, so one call per document does the job.
Parsing first means paying to convert every page of every document to text, then paying again to extract
from that text, for output you discard. Parse first only when you also need the full document text for
something else, typically chunking it for retrieval.

→ [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)

</details>

**8.** A semantic model needs both a "ship-to region" and a "bill-to region", and there is one physical
`REGION` table. How do you model it?

<details><summary>Show answer</summary>

Define two logical tables with different names — `ship_to_region` and `bill_to_region` — both pointing at
the same `base_table`. Logical table names must be unique; the underlying physical table need not be.
This is role-playing, and it is the documented pattern. Trying instead to model it with one logical table
and two relationships leaves Analyst with no way to tell the two roles apart in a question.

→ [Semantic view YAML specification](https://docs.snowflake.com/en/user-guide/views-semantic/semantic-view-yaml-spec)

</details>

**9.** A team wants a search service refreshed within a minute of new data, and is surprised by the bill.
Which `TARGET_LAG` trade-off are they making?

<details><summary>Show answer</summary>

A short `TARGET_LAG` means the service checks and refreshes often, and every refresh costs warehouse
compute to run the source query plus embedding tokens for every changed row. A one-minute lag on a table
with steady inserts pays that repeatedly, all day. The question worth asking is how stale the index is
allowed to be for the actual use case — support agents searching yesterday's tickets rarely need
sixty-second freshness, and an hour's lag can cut the refresh cost by a large multiple.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>

**10.** For a dashboard that answers five fixed business questions, would you build a semantic view for
Cortex Analyst or write the five queries by hand? What does each choice cost?

<details><summary>Show answer</summary>

Five fixed questions are five queries. Analyst earns its keep when the questions are *not* fixed — when
users will ask things you did not anticipate — and that flexibility costs you a semantic model to build
and keep aligned with the schema, plus a per-message charge on every question including the ones a cached
dashboard would have answered for free. The middle path is worth naming: build the semantic view, and
declare the five known questions as verified queries so the common cases are both fast and correct.

→ [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)

</details>

**11.** You have hour-long support calls and want to find the moment a customer mentions cancelling.
Which function, and why not the other one?

<details><summary>Show answer</summary>

`AI_MULTI_EMBED` returns an array of per-segment embeddings with `start_sec` and `end_sec`, so a match
points at a timestamp inside the call. `AI_EMBED` returns a single vector for the whole file, which can
tell you *which* call is relevant but not *where* in it. The alternative route is `AI_TRANSCRIBE` with
`timestamp_granularity`, then ordinary text search over the segments — often cheaper, and it leaves you
with a transcript you can also classify and summarise.

→ [AI_MULTI_EMBED](https://docs.snowflake.com/en/sql-reference/functions/ai_multi_embed)

</details>

**12.** *Connecting to another domain.* Your search service is built over a table containing customer
names, and a support analyst with no `SELECT` on that table can query the service. Is that a
misconfiguration, and what would you change?

<details><summary>Show answer</summary>

It is documented behaviour, not a misconfiguration: Cortex Search services search with **owner's
rights**, so `USAGE` on the service is enough to see what the owner can see. The grant model will not
protect the names. The fix belongs to the AI functions in 2.1 and the governance features in Domain 3:
build the service over `AI_REDACT(ticket_text)` so the indexed text never contained the names, and apply
masking or row-access policies to the base table for everyone else. Deciding this *before* the service is
created matters, because changing the indexed column means rebuilding the whole index.

→ [Query a Cortex Search service](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/query-cortex-search-service)

</details>
